# 揽宝智能投研交易平台 - 快速入门

本 Notebook 介绍如何使用揽宝平台进行量化策略研究。

## 1. 环境设置

In [ ]:
# 导入必要的库
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 揽宝模块
from lanbao_data.tushare_adapter import TushareAdapter
from lanbao_data.duckdb_storage import DuckDBStorage
from lanbao_strategy.strategy_template import MovingAverageCrossStrategy
from lanbao_backtest.backtest_engine import BacktestEngine, BacktestConfig

## 2. 数据获取

In [1]:
# 初始化Tushare适配器
# 注意: 需要设置 TUSHARE_TOKEN 环境变量
tushare = TushareAdapter()

# 获取股票日线数据
symbol = "000001.SZ"  # 平安银行
data = tushare.get_daily_data(symbol, start_date="20230101", end_date="20241231")

print(f"获取到 {len(data)} 条数据")
data.tail(10)

NameError: name 'TushareAdapter' is not defined

## 3. 数据存储

In [ ]:
# 初始化DuckDB存储
storage = DuckDBStorage("../data/lanbao.duckdb")

# 保存数据
storage.save_daily_data(symbol, data)

# 查询数据
stored_data = storage.get_daily_data(symbol, start_date="2024-01-01")
print(f"存储的数据: {len(stored_data)} 条")

## 4. 策略开发

In [ ]:
# 创建双均线交叉策略
strategy = MovingAverageCrossStrategy(
    strategy_id="ma_demo",
    name="演示策略",
    fast_period=5,
    slow_period=20
)

# 分析数据
analysis = strategy.analyze(data)
print(f"趋势: {analysis['trend']}，强度: {analysis['trend_strength']:.2%}")

# 生成信号
signals = strategy.generate_signals(data)
print(f"生成 {len(signals)} 个信号")
for signal in signals[-5:]:
    print(f"  {signal.timestamp}: {signal.action} - {signal.reason}")

## 5. 回测

In [ ]:
# 配置回测引擎
config = BacktestConfig(
    initial_capital=100000,
    commission_rate=0.0003,
    slippage=0.001
)

engine = BacktestEngine(config)

# 定义信号生成器
def ma_cross_signal(data):
    df = data.copy()
    df['ma5'] = df['close'].rolling(5).mean()
    df['ma20'] = df['close'].rolling(20).mean()
    
    signal = pd.Series(0, index=df.index)
    signal[df['ma5'] > df['ma20']] = 1
    signal[df['ma5'] < df['ma20']] = -1
    return signal

# 运行回测
result = engine.run_backtest(
    strategy_id="ma_cross",
    symbol=symbol,
    data=data,
    signal_generator=ma_cross_signal
)

# 查看结果
print(f"回测ID: {result.backtest_id}")
print(f"总收益: {result.total_return:.2%}")
print(f"年化收益: {result.annual_return:.2%}")
print(f"夏普比率: {result.sharpe_ratio:.2f}")
print(f"最大回撤: {result.max_drawdown:.2%}")
print(f"胜率: {result.win_rate:.2%}")
print(f"交易次数: {result.total_trades}")

## 6. 可视化

In [ ]:
# 绘制权益曲线
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 权益曲线
ax1 = axes[0]
result.equity_curve.plot(ax=ax1, label='Equity')
ax1.axhline(y=config.initial_capital, color='gray', linestyle='--', label='Initial')
ax1.set_title('Backtest Equity Curve')
ax1.set_xlabel('Date')
ax1.set_ylabel('Capital')
ax1.legend()
ax1.grid(True)

# 价格和均线
ax2 = axes[1]
data['close'].plot(ax=ax2, label='Close')
data['close'].rolling(5).mean().plot(ax=ax2, label='MA5')
data['close'].rolling(20).mean().plot(ax=ax2, label='MA20')
ax2.set_title('Price and Moving Averages')
ax2.set_xlabel('Date')
ax2.set_ylabel('Price')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. 绩效分析

In [ ]:
from lanbao_backtest.performance_analyzer import PerformanceAnalyzer

analyzer = PerformanceAnalyzer()
analysis = analyzer.analyze(result)

# 打印分析报告
print("=" * 50)
print("回测绩效分析报告")
print("=" * 50)

print("\n【基本信息】")
for k, v in analysis['summary'].items():
    print(f"  {k}: {v}")

print("\n【收益指标】")
for k, v in analysis['returns'].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2%}" if 'return' in k else f"  {k}: {v:.4f}")

print("\n【风险指标】")
for k, v in analysis['risk'].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2%}" if ('drawdown' in k) or ('volatility' in k) else f"  {k}: {v:.2f}")

print("\n【交易统计】")
for k, v in analysis['trades'].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2%}" if 'rate' in k else f"  {k}: {v:.2f}")
    else:
        print(f"  {k}: {v}")

## 8. 多策略对比

In [ ]:
# 运行多个策略
strategies = {
    'MA5/20': ma_cross_signal,
}

results = {}
for name, signal_func in strategies.items():
    result = engine.run_backtest(
        strategy_id=f"demo_{name}",
        symbol=symbol,
        data=data,
        signal_generator=signal_func
    )
    results[name] = result

# 对比
comparison = pd.DataFrame({
    name: {
        '总收益': r.total_return,
        '年化收益': r.annual_return,
        '夏普比率': r.sharpe_ratio,
        '最大回撤': r.max_drawdown,
        '胜率': r.win_rate,
        '交易次数': r.total_trades
    }
    for name, r in results.items()
}).T

print(comparison)

# 可视化对比
fig, ax = plt.subplots(figsize=(10, 6))
for name, result in results.items():
    result.equity_curve.plot(ax=ax, label=name)
ax.axhline(y=config.initial_capital, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Strategy Comparison')
ax.set_xlabel('Date')
ax.set_ylabel('Capital')
ax.legend()
ax.grid(True)
plt.show()

## 9. 保存结果

In [ ]:
# 保存回测结果
storage.save_backtest_result({
    'backtest_id': result.backtest_id,
    'strategy_id': result.strategy_id,
    'symbol': result.symbol,
    'start_date': result.start_date,
    'end_date': result.end_date,
    'total_return': result.total_return,
    'annual_return': result.annual_return,
    'sharpe_ratio': result.sharpe_ratio,
    'max_drawdown': result.max_drawdown,
    'volatility': result.volatility,
    'win_rate': result.win_rate,
    'trades_count': result.total_trades,
    'params': strategy.get_params()
})

# 查询历史回测
backtests = storage.get_backtest_results()
print(f"历史回测记录: {len(backtests)} 条")
backtests.head()

## 10. 生成报告

In [ ]:
# 生成HTML报告
html_report = analyzer.generate_report_html(result)

# 保存报告
report_path = f"../reports/backtest_{result.backtest_id}.html"
import os
os.makedirs("../reports", exist_ok=True)
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html_report)

print(f"报告已保存: {report_path}")

---

## 下一步

- 尝试修改策略参数，观察对回测结果的影响
- 开发自己的策略，继承 `StrategyTemplate` 类
- 使用ROS2节点进行实时数据处理和策略执行